In [2]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../dataset/merged_dataset.parquet")
print("Infinite Values:")
print(np.isinf(df.select_dtypes(include=np.number)).sum().sum())

Infinite Values:
0


In [3]:
negative = (df.select_dtypes(include=np.number) < 0).sum()

negative[negative > 0]


Flow Duration              88
Flow Bytes/s               59
Flow Packets/s             88
Flow IAT Mean              88
Flow IAT Max               88
Flow IAT Min             2808
Fwd IAT Min                17
Fwd Header Length          35
Bwd Header Length          22
Init Fwd Win Bytes     950431
Init Bwd Win Bytes    1179412
Fwd Seg Size Min           35
dtype: int64

In [4]:
constant_columns = [col for col in df.columns if df[col].nunique() == 1]

print("Constant Columns:", constant_columns)

Constant Columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [5]:
df.nunique().sort_values()

df.memory_usage(deep=True).sum() / 1024**2

np.float64(580.6609153747559)

In [ ]:
constant_columns = [col for col in df.columns if df[col].nunique() == 1]

print(constant_columns)

df = df.drop(columns=constant_columns)

print(df.shape)


[]
(2313810, 70)
[]


In [7]:
df.to_parquet("../dataset/cleaned_dataset.parquet", index=False)

In [8]:
df.select_dtypes(include=np.number).min().sort_values()

Fwd Header Length     -3.221223e+10
Bwd Header Length     -1.073741e+09
Fwd Seg Size Min      -5.368707e+08
Flow Bytes/s          -2.610000e+08
Flow Packets/s        -2.000000e+06
                           ...     
Idle Max               0.000000e+00
Bwd Packets/s          0.000000e+00
Fwd IAT Total          0.000000e+00
Total Fwd Packets      1.000000e+00
Subflow Fwd Packets    1.000000e+00
Length: 69, dtype: float64

In [10]:
negative_counts = (df.select_dtypes(include=np.number) < 0).sum()

negative_counts = negative_counts[negative_counts > 0]

negative_counts.sort_values(ascending=False)
for col in negative_counts.index:
    print("=" * 60)
    print(col)
    print(df[df[col] < 0][col].value_counts().head(10))

Flow Duration
Flow Duration
-1     82
-12     2
-2      2
-4      1
-13     1
Name: count, dtype: int64
Flow Bytes/s
Flow Bytes/s
-1.200000e+07    50
-6.000000e+06     2
-1.000000e+06     1
-1.930000e+08     1
-8.000000e+06     1
-6.666667e+05     1
-4.615385e+05     1
-4.000000e+06     1
-2.610000e+08     1
Name: count, dtype: int64
Flow Packets/s
Flow Packets/s
-2.000000e+06    82
-1.666667e+05     2
-1.000000e+06     2
-5.000000e+05     1
-1.538462e+05     1
Name: count, dtype: int64
Flow IAT Mean
Flow IAT Mean
-1.0     82
-12.0     2
-2.0      2
-4.0      1
-13.0     1
Name: count, dtype: int64
Flow IAT Max
Flow IAT Max
-1     82
-12     2
-2      2
-4      1
-13     1
Name: count, dtype: int64
Flow IAT Min
Flow IAT Min
-1     2671
-12      32
-2       27
-3       17
-4       17
-13      12
-11      12
-5       11
-10       4
-6        2
Name: count, dtype: int64
Fwd IAT Min
Fwd IAT Min
-1     15
-12     1
-8      1
Name: count, dtype: int64
Fwd Header Length
Fwd Header Length
-107

In [11]:
for col in negative_counts.index:
    print("="*60)
    print(col)
    print("Min:", df[col].min())
    print("Negative Count:", (df[col] < 0).sum())

Flow Duration
Min: -13
Negative Count: 88
Flow Bytes/s
Min: -261000000.0
Negative Count: 59
Flow Packets/s
Min: -2000000.0
Negative Count: 88
Flow IAT Mean
Min: -13.0
Negative Count: 88
Flow IAT Max
Min: -13
Negative Count: 88
Flow IAT Min
Min: -14
Negative Count: 2808
Fwd IAT Min
Min: -12
Negative Count: 17
Fwd Header Length
Min: -32212234632
Negative Count: 35
Bwd Header Length
Min: -1073741320
Negative Count: 22
Init Fwd Win Bytes
Min: -1
Negative Count: 950431
Init Bwd Win Bytes
Min: -1
Negative Count: 1179412
Fwd Seg Size Min
Min: -536870661
Negative Count: 35


In [12]:
df["Init Fwd Win Bytes"].value_counts().head(20)

Init Fwd Win Bytes
-1        950431
 8192     322516
 29200    238046
 256       89347
 0         75365
 65535     70912
 251       60538
 274       43442
 229       34031
 253       11605
 254        9190
 255        9018
 349        7082
 60         6882
 1024       6828
 237        6350
 258        5964
 235        5905
 114        5867
 351        5399
Name: count, dtype: int64

In [13]:
df["Init Bwd Win Bytes"].value_counts().head(20)

Init Bwd Win Bytes
-1        1179412
 235       177049
 229        94655
 0          85828
 256        31975
 31         18728
 60         17696
 29200      15557
 65535      14533
 946        13922
 122        11323
 939        11198
 123        11035
 360        10884
 972        10801
 119        10129
 28960       9834
 114         9721
 357         9405
 349         9371
Name: count, dtype: int64

In [14]:
columns_to_check = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Min",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Seg Size Min"
]

mask = (df[columns_to_check] < 0).any(axis=1)

print("Rows with invalid negative values:", mask.sum())
print("Percentage:", round(mask.mean() * 100, 4), "%")

Rows with invalid negative values: 2843
Percentage: 0.1229 %


In [15]:
# Save cleaned dataset
df.to_parquet("../dataset/cleaned_dataset.parquet", index=False)

print("✅ Cleaned dataset saved successfully!")
print("Final Shape:", df.shape)

✅ Cleaned dataset saved successfully!
Final Shape: (2313810, 70)


In [17]:
print(constant_columns)


[]
